# Quickstart

This page builds one small circuit, runs it, reads the result back, and checks that the result says
what it should.

It runs on the **local simulator**, which needs no access to the quantum computer itself, so the code
below works with nothing beyond a default `pip install claveles`, which brings the simulator with it;
see [Installation](installation.md). Every cell is executed when this documentation
is built, so the numbers and figures on this page are the ones the code actually produced.

The page speaks the vocabulary of continuous-variable quantum optics. A **mode** is one oscillator of
the light field — the thing a program computes with — and it carries two continuous quantities, the
**quadratures** $x$ and $p$, which is what a measurement reads. If that is new,
[Introduction to Quantum States](states.md) and [Introduction to Measurement](measurement.md) start
from the beginning and need nothing from the SDK.

The same kind of circuit runs on **the machine** — the optical quantum computer, not a simulation of
it. Once this page makes sense,
[Executing Circuit](execution.ipynb) is the next step.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## Three representations of a circuit

Claveles expresses a program in one of three representations. Which one you write depends on where the
program runs, and you will normally write the first of them.

| Representation | What it is | Where you write one |
| --- | --- | --- |
| **Circuit** | A continuous-variable circuit of modes and the operations applied to them, the form closest to how a circuit is usually drawn. See [Constructing Circuit Representation](circuit_repr.ipynb). | Almost always. |
| **Graph** | An intermediate form that has already been laid out on the machine's two-dimensional lattice of macronodes. See [Constructing Graph Representation](graph_repr.ipynb). | When you want to control that layout yourself. |
| **Machinery** | The measurement angles the machine is actually driven with. See [Constructing Machinery Representation](machinery_repr.ipynb). | When you want to control the machine directly. |

* **On the machine**, execution needs a machinery representation. You do not have to write one: a
  circuit representation is converted to a graph representation and then to a machinery representation
  for you.
* **On the simulator**, all three run. A circuit representation runs as it is, with no conversion at
  all, and it can hold states the machine cannot prepare, such as Fock or GKP states.

### The same program in two of them

The image below is a circuit representation of two modes and five operations:

1. Rotate mode `0` by $\frac{\pi}{4}$
2. Displace mode `0` by the complex amplitude $\alpha = 1 - i$
3. Perform a controlled-Z operation between mode `0` and mode `1`
4. Measure mode `0` at a measurement angle of $\frac{\pi}{2}$
5. Measure mode `1` at a measurement angle of $0$

See [Gates & Operations](primitives/gates.md) for what each of these operations does, and [Measurements](primitives/measurements.md) for the measurement itself.

```{image} _images/circuit_repr_sample_circuit_light.svg
:alt: circuit
:class: only-light
```
```{image} _images/circuit_repr_sample_circuit_dark.svg
:alt: circuit
:class: only-dark
```

Below is the graph representation of that same program. The twelve large circles are **macronodes**,
and the four small circles inside each one are **micronodes**. A column of vertically aligned
macronodes is a **step**, so this graph has `3` local macronodes per step and `4` steps.

The blue micronodes carry mode `0`, which is transformed as it is teleported through four macronodes;
the red ones carry mode `1`, which passes through three. Both modes are initialized at the macronode
they start from and measured at the one they end at.

![graph](_images/quickstart_graph.svg)

The rest of this page stays in the circuit representation.

## Building a circuit

The circuit we will run rotates one mode and measures it:


In [ ]:
from math import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops import std
from claveles.circuit.state import QuantumState
from claveles.circuit.visualize import make_figure

circuit = CircuitRepr("quickstart")
circuit.Q(0) | QuantumState.squeezed(r=1.0, phi=0.0)  # the input state, x-squeezed
circuit.Q(0) | std.PhaseRotation(pi / 4)
circuit.Q(0) | std.MeasureHomodyne(pi / 4)

make_figure(circuit);

Three things are worth reading off that cell.

* {py:class}`~claveles.circuit.CircuitRepr` holds the program, {py:meth}`~claveles.circuit.CircuitRepr.Q`
  names a mode, and `|` applies something to it. Operations come from
  {py:mod}`claveles.circuit.ops.std`, here {py:class}`~claveles.circuit.ops.std.PhaseRotation` and
  {py:class}`~claveles.circuit.ops.std.MeasureHomodyne`.
* A **state is placed on the mode first**. The simulator has no squeezing level to invent, so it wants
  a state on every mode; the machine supplies its own squeezed states, so there you would leave the
  mode empty instead, or place {py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed` to
  choose only their angle. {ref}`sec:circuit-placing-states` covers both, and
  [Working with Quantum States](quantum_states.md) covers the state objects.
* The state placement is an operation in the sequence like any other, so the figure draws it as the
  $|\psi\rangle$ box at the left of the mode line — a mode left empty has none. See
  {ref}`sec:circuit-drawing` for how to read a figure.

## Running it

A **client** decides where a program runs. {py:class}`~claveles.client.SimulatorClient` runs it here, in
this process, and {py:attr}`~claveles.client.abstract.AbstractClient.n_shots` says how many times:

In [ ]:
from claveles.client import SimulatorClient
from claveles.execute import execute

client = SimulatorClient(n_shots=1000)
result = execute(circuit, client)

{py:func}`~claveles.execute.execute` takes any representation and any client, and hands back an
{py:class}`~claveles.execute.ExecutionResult`. Swapping the client for an
{py:class}`~claveles.client.OptQCClient` runs it on the machine instead, once the state on mode `0`
is one the machine takes, as above.

## Reading the result

Indexing the result gives one shot, and a shot lists what each measurement read:

In [ ]:
print("shots            =", result.n_shots)
print("shot 0           =", result[0])
print("mode 0 of shot 0 =", result[0][0].mode, "->", result[0][0].value)

Each entry is a {py:class}`~claveles.circuit.result.CircuitModeMeasuredValue`, carrying the
{py:attr}`~claveles.circuit.result.CircuitModeMeasuredValue.mode` it read and the
{py:attr}`~claveles.circuit.result.CircuitModeMeasuredValue.value` it got. A measurement consumes the mode
it reads, so a mode appears at most once in a shot.

That is the whole result for this circuit. A simulation can also hand back the quantum state each shot
left behind, and the timings and other per-client details live on
{py:attr}`~claveles.execute.ExecutionResult.client_result`; both are covered in
[Simulating Circuit](simulation.ipynb).

## Checking that the rotation happened

One shot of a {ref}`homodyne measurement <sec:measurement-homodyne>` is a random number, so a single
value says little. The **variance** over many shots does: it is the width of the state along the
direction that was measured.

The input state is {ref}`squeezed <sec:states-squeezed>` along $x$, which makes it narrow in $x$ and
broad in $p$. Rotating it by $\frac{\pi}{4}$ turns the narrow direction to $\frac{\pi}{4}$ as well.
So sweeping the measurement angle $\phi$ over $0, \frac{\pi}{8}, \frac{\pi}{4}, \dots, \pi$ should
give the smallest variance at $\phi = \frac{\pi}{4}$ and the largest a quarter turn away, at
$\phi = \frac{3\pi}{4}$.

```{image} _images/quickstart_multi_param_light.svg
:alt: quickstart_multi_param
:class: only-light
```
```{image} _images/quickstart_multi_param_dark.svg
:alt: quickstart_multi_param
:class: only-dark
```

In [ ]:
import numpy as np


def build_circuit(phi: float) -> CircuitRepr:
    # The circuit above, measured at angle `phi` instead.
    c = CircuitRepr(f"phi={phi}")
    c.Q(0) | QuantumState.squeezed(r=1.0, phi=0.0)
    c.Q(0) | std.PhaseRotation(pi / 4)
    c.Q(0) | std.MeasureHomodyne(phi)
    return c


angle_list = np.linspace(0, pi, 9)
var_list = []

for phi in angle_list:
    shots = execute(build_circuit(phi), client)
    # The measured value of mode 0 in every shot.
    mode0_values = [smv[0].value for smv in shots]
    var_list.append(np.var(mode0_values))

print(np.round(var_list, 3))

The variance is smallest at $\frac{\pi}{4}$ and largest at $\frac{3\pi}{4}$, as expected — that is the
check. The exact values follow from the squeezing parameter, if you want them: for $r = 1$ the two
extremes are $e^{-2r}/2 \approx 0.068$ and $e^{2r}/2 \approx 3.69$, in units where the vacuum variance
is $\frac{1}{2}$; the sampled values above sit near those. Plotted:

In [ ]:
import matplotlib.pyplot as plt

angle_labels = [
    r"$0$",
    r"$\frac{\pi}{8}$",
    r"$\frac{\pi}{4}$",
    r"$\frac{3\pi}{8}$",
    r"$\frac{\pi}{2}$",
    r"$\frac{5\pi}{8}$",
    r"$\frac{3\pi}{4}$",
    r"$\frac{7\pi}{8}$",
    r"$\pi$",
]

plt.figure(figsize=(8, 5))
plt.plot(angle_list, var_list, "ro-", label="Mode 0")
plt.axhline(0.5, color="gray", linestyle=":", label="vacuum")
plt.xlabel("Measurement angle")
plt.ylabel("Variance")
plt.xticks(angle_list, angle_labels)
plt.legend()
plt.show()

## Where to go next

* [Constructing Circuit Representation](circuit_repr.ipynb) — the rest of what a circuit can hold:
  two-mode gates, feedforward, composition, saving and loading.
* [Working with Quantum States](quantum_states.md) — the state objects, and what comes back from a
  simulation.
* [Simulating Circuit](simulation.ipynb) — the simulator's own options: Fock or bosonic states, saved
  states, Wigner functions.
* [Executing Circuit](execution.ipynb) — the same circuit on the machine.
* [Gates & Operations](primitives/gates.md) — every operation, with its definition and its Heisenberg action.